[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/91_remainder_walk_bd260730_solution.ipynb)

# 参考解法：余数行走

Reference solution.

## 解析

**结论：把轨迹点转成前缀余数序列，问题化为「每个起点向右最长不重复段」，用滑动窗口一次线性求和。**

### 前缀余数
在倍长数组（`a` 接一份 `a`）上定义 `P[0]=0`，`P[i]=(P[i-1]+a[(i-1)%n]) % M`。起点 `s` 的第 `t` 个轨迹点`S_t = (P[s+t]-P[s]) mod M`。固定 `s` 时 `P[s]` 是常数，故 `S_1..S_t` 两两不同 ⇔ `P[s+1..s+t]` 两两不同。

### 滑动窗口
左边界 `j` 从 1 到 n（即 `s+1`），右指针 `r` 单调不回退：不断把 `P[r]` 加入 `seen` 直到遇到重复或触达 `j+n-1` 上限，此时 `L_s = r - j`。移动左边界时把 `P[j]` 移出 `seen`。总复杂度 `O(n)`。

### 一个易错点
轨迹点不含 `S_0`（即不含 `P[s]` 自身），所以窗口和判重都从 `j = s+1` 起、不包含 `P[s]`。

### 验证
已用直接逐点累加的暴力实现在数千组随机 `(n, M, a)`（含负数、`M=1`）上对拍一致，并复现官方样例。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
from typing import List

In [ ]:
# ✅ SOLUTION

from typing import List

class Solution:
    def total_length(self, n: int, M: int, a: List[int]) -> int:
        P = [0] * (2 * n + 1)
        for i in range(1, 2 * n + 1):
            P[i] = (P[i - 1] + a[(i - 1) % n]) % M
        total = 0
        seen = set()
        r = 1                       # monotonic right pointer
        for j in range(1, n + 1):   # j = s + 1 is the left boundary
            if r < j:
                r = j
                seen = set()
            cap = j + n - 1         # at most n distinct points
            while r <= cap and P[r] not in seen:
                seen.add(P[r])
                r += 1
            total += (r - j)
            seen.discard(P[j])
        return total

In [ ]:
sol = Solution()
print(sol.total_length(5, 3, [1, 2, 2, 1, 2]))   # 11
print(sol.total_length(4, 5, [5, 0, 5, 0]))      # 4

In [ ]:
from torch_judge import check
check('remainder_walk_bd260730')